In [1]:
#Purpose
#analysing rising sea level trends in the Pacific and assessing the projected impact on low-lying nations

#Step 0 Importing Libraries
library(tidyverse)
library(ggplot2)
library(reshape2)

Warning message:
"package 'ggplot2' was built under R version 4.5.3"
── Attaching core tidyverse packages ──────────────────
✔ dplyr     1.2.0     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.2     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.1     
── Conflicts ───────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Warning message:
"package 'reshape2' was built under R version 4.5.3"

Attaching package: 'reshape2'


The following object is masked from 'package:tidyr':

    smiths




In [2]:
#Step 1 Importing dataset and inspection
main_df <- read.csv("Honululu.csv")

ERROR: Error in read.table(file = file, header = header, sep = sep, quote = quote, : more columns than column names


In [3]:
?read.csv #Going through the arguments section to check how to handle the column name error

read.table {utils},R Documentation
file,"the name of the file which the data are to be read from. Each row of the table appears as one line of the file. If it does not contain an absolute path, the file name is relative to the current working directory, getwd(). Tilde-expansion is performed where supported. This can be a compressed file (see file). Alternatively, file can be a readable text-mode connection (which will be opened for reading if necessary, and if so closed (and hence destroyed) at the end of the function call). (If stdin() is used, the prompts for lines may be somewhat confusing. Terminate input with a blank line or an EOF signal, Ctrl-D on Unix and Ctrl-Z on Windows. Any pushback on stdin() will be cleared before return.) file can also be a complete URL. (For the supported URL schemes, see the ‘URLs’ section of the help for url.)"
header,"a logical value indicating whether the file contains the names of the variables as its first line. If missing, the value is determined from the file format: header is set to TRUE if and only if the first row contains one fewer field than the number of columns."
sep,"the field separator character. Values on each line of the file are separated by this character. If sep = """" (the default for read.table) the separator is ‘white space’, that is one or more spaces, tabs, newlines or carriage returns."
quote,"the set of quoting characters. To disable quoting altogether, use quote = """". See scan for the behaviour on quotes embedded in quotes. Quoting is only considered for columns read as character, which is all of them unless colClasses is specified."
dec,the character used in the file for decimal points.
numerals,"string indicating how to convert numbers whose conversion to double precision would lose accuracy, see type.convert. Can be abbreviated. (Applies also to complex-number inputs.)"
row.names,"a vector of row names. This can be a vector giving the actual row names, or a single number giving the column of the table which contains the row names, or character string giving the name of the table column containing the row names. If there is a header and the first row contains one fewer field than the number of columns, the first column in the input is used for the row names. Otherwise if row.names is missing, the rows are numbered. Using row.names = NULL forces row numbering. Missing or NULL row.names generate row names that are considered to be ‘automatic’ (and not preserved by as.matrix)."
col.names,"a vector of optional names for the variables. The default is to use ""V"" followed by the column number."
as.is,"controls conversion of character variables (insofar as they are not converted to logical, numeric or complex) to factors, if not otherwise specified by colClasses. Its value is either a vector of logicals (values are recycled if necessary), or a vector of numeric or character indices which specify which columns should not be converted to factors. Note: to suppress all conversions including those of numeric columns, set colClasses = ""character"". Note that as.is is specified per column (not per variable) and so includes the column of row names (if any) and any columns to be skipped."
tryLogical,"a logical determining if columns consisting entirely of ""F"", ""T"", ""FALSE"", and ""TRUE"" should be converted to logical; passed to type.convert, true by default."


In [4]:
main_df <- read.csv("Honululu.csv", skip = 5, fill = TRUE, row.names = NULL) #CSV file containing Relative Sea Level Trends for Honolulu, Hawaii.
#Used skip function because CSV had some text and a blank space from Line 1-5 which was causing an error to occur and not allow to be read properly
#Fill = TRUE handled the trailing comma present on every data row, which was causing R to count 7 fields per row against only 6 column headers.
#row.names = NULL prevents R from misinterpreting the first column as row labels.

In [5]:
glimpse(main_df) #Transposed view of entire dataset, running horizontal instead of vertical

Rows: 1,453
Columns: 7
$ row.names    <chr> "1905", "1905", "1905", "1905", "1905", "1905", "1905", "…
$ Year         <int> 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 1, 2, 3, 4, 5, 6, …
$ Month        <dbl> -0.120, -0.119, -0.106, -0.133, -0.115, -0.161, -0.176, -…
$ Monthly_MSL  <dbl> -0.125, -0.125, -0.125, -0.125, -0.125, -0.125, -0.124, -…
$ Linear_Trend <dbl> -0.111, -0.111, -0.111, -0.111, -0.111, -0.111, -0.110, -…
$ High_Conf.   <dbl> -0.139, -0.139, -0.139, -0.139, -0.139, -0.139, -0.138, -…
$ Low_Conf.    <lgl> NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, N…


In [6]:
head(main_df) #First 5 rows of the Relative Sea Level Trends for Honolulu, Hawaii

,row.names,Year,Month,Monthly_MSL,Linear_Trend,High_Conf.,Low_Conf.
,<chr>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<lgl>
1,1905,1,-0.120,-0.125,-0.111,-0.139,NA
2,1905,2,-0.119,-0.125,-0.111,-0.139,NA
3,1905,3,-0.106,-0.125,-0.111,-0.139,NA
4,1905,4,-0.133,-0.125,-0.111,-0.139,NA
5,1905,5,-0.115,-0.125,-0.111,-0.139,NA
6,1905,6,-0.161,-0.125,-0.111,-0.139,NA


In [ ]:
#Step 1.1 Transforming the dataset from long to wide using tidyr from tidyverse library
pivot_wider(main_df,
            id_cols = Year, #Choosing First.Year as to which each station first began recording.
            id_expand = FALSE, #By setting FALSE, it doesn't result into more rows
            names_from = Station.Name, #Column is named Station.Name
            values_from = MSL.Trends..mm.yr., #Retrieves values from MSL.Trends..mm.yr. column. This is the Sea Level values recorded and places it under each Station.Name column
            names_prefix = "", #No names_prefix
            names_sep = "_", #No joining values or variables into a single string as a column name
            names_glue = NULL, #No custom column name
            names_sort = TRUE, #Column names are ordered by first appearance
            names_vary  = "fastest",
            names_expand = FALSE, #No extra columns required from possible values in names_from
            names_repair = "check_unique", #Checks error if the columns are duplicated
            values_fill = NULL, #CSV has already set values
            values_fn = NULL, 
            unused_fn = NULL #No need to summarise values from unused columns or dropping any unused columns
)